## Data Pre-Processing and Modeling

In [85]:
import pandas as pd
import numpy as np
import re
from nltk.stem import SnowballStemmer
from nltk.tokenize import word_tokenize

In [86]:
df = pd.read_csv("/content/app_reviews_labeled.csv")

In [87]:
df.shape

(50000, 4)

In [88]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   content        49992 non-null  object
 1   score          50000 non-null  int64 
 2   thumbsUpCount  50000 non-null  int64 
 3   label          50000 non-null  object
dtypes: int64(2), object(2)
memory usage: 1.5+ MB


In [89]:
df.isnull().sum()

,0
content,8
score,0
thumbsUpCount,0
label,0


In [90]:
df.dropna(inplace=True)

In [91]:
df.drop('score', axis=1, inplace=True)

In [92]:
df.sample(6)

,content,thumbsUpCount,label
31004,Omg what a great app😱,0,positive
25903,"I strongly dislike emojis, so when this was pr...",9,negative
19575,It's a good app but it regularly misses to tra...,0,negative
12372,Glitchy on every corporate device that I've ev...,52,negative
36423,Make it like samsung notes it has more feature...,0,positive
10048,"Okay, short and sweet, I HATE this new tab upd...",3,negative


In [93]:
# Convert the sentiment into the numbers
def sentiment_into_number(sentiment):
  if sentiment == 'negative':
    return 0
  elif sentiment == 'neutral':
    return 1
  elif sentiment == 'positive':
    return 2

df['label'] = df['label'].apply(sentiment_into_number)

In [94]:
df.head()

,content,thumbsUpCount,label
0,Working with this app is so difficult. Default...,566,0
1,I would give it 0 stars if possible. No option...,189,0
2,Dear Google.. I found a very critical bug..cus...,105,0
3,Worst interface ever......can't even add a new...,403,0
4,"While opening a saved contact entry, this app ...",277,1


In [95]:
import nltk
nltk.download('punkt')
import re
import unicodedata
from nltk.stem import PorterStemmer

def lower_text(text):
  """
  This function is use to lower the playstore reviews content/text
  """
  return text.lower()


def clean_text(text):
    """
    Clean Play Store review text.

    Removes:
    - HTML tags
    - URLs
    - Emojis
    - Extra whitespace
    """

    text = str(text)

    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Remove emojis and other Unicode symbols
    emoji_pattern = re.compile(
        "["
        "\U0001F300-\U0001F5FF"  # Miscellaneous Symbols and Pictographs
        "\U0001F600-\U0001F64F"  # Emoticons
        "\U0001F680-\U0001F6FF"  # Transport & Map Symbols
        "\U0001F700-\U0001F77F"  # Alchemical Symbols
        "\U0001F780-\U0001F7FF"  # Geometric Shapes Extended
        "\U0001F800-\U0001F8FF"  # Supplemental Arrows-C
        "\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
        "\U0001FA00-\U0001FA6F"  # Chess Symbols
        "\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
        "\U00002702-\U000027B0"  # Dingbats
        "\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE,
    )

    text = emoji_pattern.sub(" ", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text)

    # Remove leading/trailing whitespace
    text = text.strip()

    return text

def apply_stemming(text):
    """Apply Snowball stemming to a sentence."""
    stemmer = PorterStemmer()
    words = word_tokenize(text)
    stemmed_words = [stemmer.stem(word) for word in words]
    return " ".join(stemmed_words)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [96]:
import nltk
nltk.download('punkt_tab', quiet=True)


def text_preprocessing(text):
  text_lower = lower_text(text)
  cleaned_content = clean_text(text_lower)
  # stemmed_text = apply_stemming(cleaned_content)

  return cleaned_content

df['content'] = df['content'].apply(text_preprocessing)

In [97]:
from sklearn.model_selection import train_test_split

X = df["content"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [98]:
X_train.shape

(39993,)

In [99]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [100]:
from sklearn.metrics import classification_report
from sklearn.svm import LinearSVC

# Initialize model with your specific parameters
model = LinearSVC(
    C=1.1709879033298216,
    tol=0.000008137560594377145,
    loss='hinge',
    fit_intercept=True,
    class_weight='balanced',
    max_iter=4324
)

# Train
model.fit(X_train_tfidf, y_train)

# Predict
y_pred = model.predict(X_test_tfidf)

# Classification report
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.85      0.89      0.87      4436
           1       0.74      0.71      0.73      2148
           2       0.90      0.87      0.88      3415

    accuracy                           0.84      9999
   macro avg       0.83      0.82      0.83      9999
weighted avg       0.84      0.84      0.84      9999



In [101]:
sample = pd.DataFrame({
    "content": [
        "Amazing app!",
        "Too many bugs and crashes every day.",
        "It is okay, nothing special.",
        "Beautiful app",
        "Absolutely love this! Best user experience ever.",
        "The recent update completely broke the login screen.",
        "It does what it says, but the UI could be better.",
        "Highly recommended! Saves me so much time daily.",
        "Total waste of time. It freezes constantly on my phone.",
        "Just downloaded it. It works fine for now.",
        "Incredibly fast and very intuitive to navigate.",
        "Extremely disappointed. Terrible customer support.",
        "An average application, standard features like others.",
        "Perfect tool! I cannot imagine my routine without it."
    ]
})

# 4. Preprocess text into a separate column (FIXED 'df' error and preserved original text)
sample['content_clean'] = sample['content'].apply(text_preprocessing)

# 5. Transform using the existing fitted vectorizer
content_tfidf = tfidf_vectorizer.transform(sample['content_clean'])

# 6. Predict and append labels (FIXED model reference)
sample['label'] = model.predict(content_tfidf)

# 7. Display results side-by-side
print("\nPredicted Sample Sentiments:")


Predicted Sample Sentiments:


In [102]:
sample[['content', 'label']]

,content,label
0,Amazing app!,2
1,Too many bugs and crashes every day.,0
2,"It is okay, nothing special.",1
3,Beautiful app,2
4,Absolutely love this! Best user experience ever.,2
5,The recent update completely broke the login s...,0
6,"It does what it says, but the UI could be better.",2
7,Highly recommended! Saves me so much time daily.,2
8,Total waste of time. It freezes constantly on ...,0
9,Just downloaded it. It works fine for now.,2


In [103]:
# 2. Define the mapping dictionary
class_map = {0: 'negative', 1: 'neutral', 2: 'positive'}

# 3. Create DataFrame preserving the original X_test index
df_results = pd.DataFrame({
    'Text_Content': X_test,
    'Predicted_Label': y_pred
})

# 4. Map the numeric labels to text sentiments
df_results['Predicted_Label'] = df_results['Predicted_Label'].map(class_map)

In [104]:
percentages = df_results['Predicted_Label'].value_counts(normalize=True) * 100

In [105]:
percentages

,proportion
Predicted_Label,
negative,46.134613
positive,33.073307
neutral,20.792079


## Clustering

In [106]:
!pip install sentence-transformers umap-learn hdbscan keybert -q

In [107]:
df_predicted = pd.DataFrame({
    'content': X_test,
    'label': y_pred
})

# 2. Define the mapping dictionary
class_map = {0: 'negative', 1: 'neutral', 2: 'positive'}

df_predicted['sentiment'] = df_predicted['label'].map(class_map)

In [108]:
df_predicted.sample(5)

,content,label,sentiment
45082,playboy,1,neutral
41524,it has been two days end to end but it is not ...,0,negative
25351,ok,2,positive
27772,what is this pop-up suggestion/dictionary thin...,0,negative
13227,"absolutely horrid super glitchy, drains batter...",0,negative


In [109]:
import pandas as pd

new_df = pd.read_csv('/content/app_reviews_labeled.csv')
new_df.sample(6)

,content,score,thumbsUpCount,label
4009,"I used to absolutely adore hooked, I just don'...",3,72,negative
3547,Describes w e your experience (optional),5,0,neutral
23810,Good,2,0,positive
18590,How I❤ This App Create and edit presentations ...,5,20,positive
40235,Nice,3,0,positive
2923,love it but i don't like the update that remov...,3,4,negative


In [110]:
# 1. DataFrame for positive label
df_positive = new_df[new_df["label"] == "positive"].nlargest(
    5, "thumbsUpCount"
)

# 2. DataFrame for neutral label
df_neutral = new_df[new_df["label"] == "neutral"].nlargest(5, "thumbsUpCount")

# 3. DataFrame for negative label
df_negative = new_df[new_df["label"] == "negative"].nlargest(
    5, "thumbsUpCount"
)

In [111]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from sentence_transformers import SentenceTransformer
import umap
import hdbscan
from keybert import KeyBERT
import json
from typing import Optional, Dict, List, Any
import hashlib

class SentimentTopicClusterer:
    """
    Clusters reviews by sentiment and extracts keywords/topics per cluster.

    Expected DataFrame columns:
        - content  : review text
        - sentiment: predicted sentiment label (e.g. "negative", "neutral", "positive")
    """

    def __init__(
        self,
        embedding_model_name: str = "all-MiniLM-L6-v2",
        top_n_keywords: int = 3,
        min_reviews_to_cluster: int = 30,
        top_topics_to_show: int = 5,          # for user-facing summary
        enable_embedding_cache: bool = True,  # in-memory cache
    ):
        self.top_n_keywords = top_n_keywords
        self.min_reviews_to_cluster = min_reviews_to_cluster
        self.top_topics_to_show = top_topics_to_show
        self.enable_embedding_cache = enable_embedding_cache

        print("Loading embedding + keyword models...")
        self.embedder = SentenceTransformer(embedding_model_name)
        self.kw_model = KeyBERT(self.embedder)

        # In-memory embedding cache: key = hash of texts → embeddings
        self._embedding_cache: Dict[str, np.ndarray] = {}

        self.df = None
        self.embeddings = None
        self.results_by_sentiment = {}
        self.topics_by_sentiment = {}
        self.df_final = None
        self.app_summary = {}
        self.user_facing_summary = {}   # clean summary for the extension

    # ------------------------------------------------------------------
    # Helper: create a stable cache key from the list of texts
    # ------------------------------------------------------------------
    def _make_cache_key(self, texts: List[str]) -> str:
        joined = "||".join(texts)
        return hashlib.md5(joined.encode("utf-8")).hexdigest()

    # ------------------------------------------------------------------
    # Main entry point
    # ------------------------------------------------------------------
    def fit(self, df: pd.DataFrame):
        """
        Run the full pipeline on the given DataFrame.
        """
        if not {"content", "sentiment"}.issubset(df.columns):
            raise ValueError("DataFrame must contain 'content' and 'sentiment' columns.")

        self.df = df.copy()
        self.df["content"] = self.df["content"].astype(str)
        texts = self.df["content"].tolist()

        # ---- Embeddings with optional caching ----
        cache_key = self._make_cache_key(texts) if self.enable_embedding_cache else None

        if self.enable_embedding_cache and cache_key in self._embedding_cache:
            print("Using cached embeddings...")
            self.embeddings = self._embedding_cache[cache_key]
        else:
            print("Embedding all reviews...")
            self.embeddings = self.embedder.encode(
                texts,
                batch_size=64,
                show_progress_bar=True,
            )
            if self.enable_embedding_cache and cache_key is not None:
                self._embedding_cache[cache_key] = self.embeddings

        self.df["_embedding_idx"] = range(len(self.df))

        # ---- Cluster per sentiment ----
        self.results_by_sentiment = {}
        self.topics_by_sentiment = {}

        for sentiment in ["negative", "neutral", "positive"]:
            clustered_df, topics = self._cluster_sentiment_group(sentiment)
            self.results_by_sentiment[sentiment] = clustered_df
            self.topics_by_sentiment[sentiment] = topics

        # ---- Combine results ----
        valid = [v for v in self.results_by_sentiment.values() if v is not None]
        self.df_final = pd.concat(valid, ignore_index=True) if valid else pd.DataFrame()

        # ---- Build keyword summary (raw) ----
        self.app_summary = {}
        for sentiment, topics in self.topics_by_sentiment.items():
            keyword_list = []
            for cluster_id, name in topics.items():
                if cluster_id == -1:
                    continue
                keyword_list.extend(name.split(" | "))
            self.app_summary[sentiment] = keyword_list

        # ---- Build clean user-facing summary (top topics + counts) ----
        self._build_user_facing_summary()

        return self

    # ------------------------------------------------------------------
    # Clustering for one sentiment
    # ------------------------------------------------------------------
    def _cluster_sentiment_group(self, sentiment_label: str):
        subset = self.df[self.df["sentiment"] == sentiment_label]
        n = len(subset)

        if n < self.min_reviews_to_cluster:
            print(
                f"[{sentiment_label}] Too few reviews ({n}) to cluster meaningfully. Skipping."
            )
            return None, {}

        sub_embeddings = self.embeddings[subset["_embedding_idx"].values]

        # ---- UMAP ----
        reducer = umap.UMAP(
            n_components=min(10, n - 2),
            n_neighbors=min(15, n - 1),
            min_dist=0.0,
            metric="cosine",
            random_state=42,
        )
        reduced = reducer.fit_transform(sub_embeddings)

        # ---- HDBSCAN ----
        min_cluster_size = int(np.clip(n * 0.02, 15, 80))
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=max(5, min_cluster_size // 5),
            metric="euclidean",
            cluster_selection_method="eom",
        )
        cluster_labels = clusterer.fit_predict(reduced)

        subset = subset.copy()
        subset["cluster"] = cluster_labels

        n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
        n_noise = (cluster_labels == -1).sum()
        print(
            f"\n[{sentiment_label.upper()}] {n} reviews -> {n_clusters} clusters, "
            f"{n_noise} noise ({n_noise / n * 100:.1f}%)"
        )

        # ---- KeyBERT keyword extraction ----
        cluster_names = {}
        for cluster_id in sorted(set(cluster_labels)):
            if cluster_id == -1:
                cluster_names[-1] = "Uncategorized"
                continue

            cluster_reviews = (
                subset[subset["cluster"] == cluster_id]["content"].tolist()
            )
            combined_text = " ".join(cluster_reviews[:200])[:20000]

            try:
                keywords = self.kw_model.extract_keywords(
                    combined_text,
                    keyphrase_ngram_range=(1, 2),
                    stop_words="english",
                    top_n=self.top_n_keywords,
                    use_mmr=True,
                    diversity=0.5,
                )
                topic_name = (
                    " | ".join([kw[0] for kw in keywords]) if keywords else "N/A"
                )
            except Exception:
                topic_name = "N/A"

            cluster_names[cluster_id] = topic_name
            print(
                f"  Cluster {cluster_id:2d} ({len(cluster_reviews):4d} reviews): {topic_name}"
            )

        subset["topic"] = subset["cluster"].map(cluster_names)
        return subset, cluster_names

    # ------------------------------------------------------------------
    # Build clean summary for the extension UI
    # ------------------------------------------------------------------
    def _build_user_facing_summary(self):
        """
        Creates a clean structure ready for the extension:

        {
          "negative": [
            {"topic": "battery life | charging", "count": 42, "percentage": 18.5},
            ...
          ],
          "neutral": [...],
          "positive": [...]
        }
        """
        self.user_facing_summary = {}

        for sentiment in ["negative", "neutral", "positive"]:
            clustered_df = self.results_by_sentiment.get(sentiment)

            if clustered_df is None or clustered_df.empty:
                self.user_facing_summary[sentiment] = []
                continue

            total = len(clustered_df)

            # Count reviews per topic (ignore Uncategorized / -1)
            topic_counts = (
                clustered_df[clustered_df["cluster"] != -1]
                .groupby("topic")
                .size()
                .reset_index(name="count")
            )

            # Sort by count descending and keep top N
            topic_counts = topic_counts.sort_values("count", ascending=False)
            topic_counts = topic_counts.head(self.top_topics_to_show)

            topics_list = []
            for _, row in topic_counts.iterrows():
                topics_list.append({
                    "topic": row["topic"],
                    "count": int(row["count"]),
                    "percentage": round(row["count"] / total * 100, 1)
                })

            self.user_facing_summary[sentiment] = topics_list

    # ------------------------------------------------------------------
    # Public helpers
    # ------------------------------------------------------------------
    def get_summary(self) -> dict:
        """Raw keyword list per sentiment (legacy)."""
        return self.app_summary

    def get_user_facing_summary(self) -> dict:
        """Clean summary for the extension UI (recommended)."""
        return self.user_facing_summary

    def print_summary(self):
        """Pretty-print the user-facing summary."""
        print(json.dumps(self.user_facing_summary, indent=2))

    def get_final_dataframe(self) -> pd.DataFrame:
        """Return the combined clustered DataFrame."""
        return self.df_final

    def clear_embedding_cache(self):
        """Clear the in-memory embedding cache if needed."""
        self._embedding_cache.clear()
        print("Embedding cache cleared.")

In [112]:
clusterer = SentimentTopicClusterer(
    top_n_keywords=3,
    top_topics_to_show=5,          # show top 5 topics
    enable_embedding_cache=True    # cache embeddings
)

clusterer.fit(df_predicted)   # df must have columns: content, sentiment

# What the extension should use:
summary = clusterer.get_user_facing_summary()
print(json.dumps(summary, indent=2))

Loading embedding + keyword models...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding all reviews...


Batches:   0%|          | 0/157 [00:00<?, ?it/s]


[NEGATIVE] 4613 reviews -> 19 clusters, 833 noise (18.1%)
  Cluster  0 ( 100 reviews): google pay | gpay useless | unable gift
  Cluster  1 ( 339 reviews): music app | playlists messed | agree frustrating
  Cluster  2 ( 111 reviews): fix youtube | dash glitchy | app updates
  Cluster  3 ( 109 reviews): fitness app | tracker struggles | steps updating
  Cluster  4 ( 103 reviews): sound alarm | spotify difficult | setting app
  Cluster  5 (  88 reviews): articles app | news frivolous | annoying widget
  Cluster  6 ( 484 reviews): pinyin shotcut | typing disappear | gboard using
  Cluster  7 ( 234 reviews): devices issues | uninstalled 2019 | google nest
  Cluster  8 ( 128 reviews): app dark | calendars settings | accessibility guide
  Cluster  9 ( 200 reviews): google map | updates working | temperature home
  Cluster 10 ( 141 reviews): glitchy email | vacuum schedule | apps services
  Cluster 11 ( 340 reviews): dialing option | app recording | fix bug
  Cluster 12 ( 469 reviews): photo

In [113]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio


# Colors
colors = {
    "negative": "#EF4444",   # red
    "neutral":  "#6B7280",   # gray
    "positive": "#22C55E"    # green
}

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=["Negative", "Neutral", "Positive"],
    horizontal_spacing=0.12
)

for i, sentiment in enumerate(["negative", "neutral", "positive"], start=1):
    data = summary[sentiment]

    # Shorten long topic names for readability
    topics = [t["topic"][:45] + "..." if len(t["topic"]) > 45 else t["topic"] for t in data]
    percentages = [t["percentage"] for t in data]
    counts = [t["count"] for t in data]

    # Reverse so highest is on top
    topics = topics[::-1]
    percentages = percentages[::-1]
    counts = counts[::-1]

    fig.add_trace(
        go.Bar(
            y=topics,
            x=percentages,
            orientation="h",
            marker_color=colors[sentiment],
            text=[f"{p}% ({c})" for p, c in zip(percentages, counts)],
            textposition="auto",
            name=sentiment.capitalize(),
            hovertemplate="<b>%{y}</b><br>%{x}% of reviews<br>Count: %{customdata}<extra></extra>",
            customdata=counts
        ),
        row=1, col=i
    )

    fig.update_xaxes(title_text="% of reviews", row=1, col=i, range=[0, max(percentages)*1.25])

fig.update_layout(
    title_text="Top Topics by Sentiment",
    height=480,
    width=1200,
    showlegend=False,
    margin=dict(l=20, r=20, t=60, b=40),
    font=dict(size=12)
)

fig.show()

# Optional: save as interactive HTML (good for extension)
# fig.write_html("topic_summary.html")

## LLM

In [114]:
!pip install langchain langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.3 MB/s eta 0:00:00


In [123]:
SYSTEMPROMPT = """You are a product analyst writing a short, human briefing about an app based on its Play Store user reviews.

You will be given:
- TOPIC_CLUSTERS: keyword clusters extracted from negative, neutral, and positive reviews, with how common each is.
- TOP_POSITIVE_REVIEWS, TOP_NEUTRAL_REVIEWS, TOP_NEGATIVE_REVIEWS: the most upvoted real reviews for each sentiment.

Do NOT just restate or list the raw clusters or reviews. Read them, understand the real patterns behind them, and write your own independent analysis in plain English.

Ignore incoherent or non-English keyword clusters — don't force meaning onto noise.

Write your response as plain text using this exact format (markdown-style headings and bullets, no JSON, no code fences):

## Overall Summary
A short 2-3 sentence paragraph giving a blunt, honest take on where this app stands overall based on what users are saying.

## What Users Like
- Short bullet describing one genuine strength, synthesized from the data
- (max 4 bullets)

## What Users Are Complaining About
- Short bullet describing one genuine, recurring problem, synthesized from the data
- (max 4 bullets)

## What's Happening in the Reviews
A short paragraph (2-4 sentences) describing the overall pattern/behavior in the feedback — e.g. is negative sentiment concentrated around one specific feature, is there a recent spike in a particular complaint, do positive and negative reviews contradict each other, etc.

## Recommendations
- Short, concrete, actionable bullet for the developers to fix or improve something
- (max 3 bullets)

Rules:
- Never output JSON, code fences, or markdown tables — only the headings and bullets described above.
- Each bullet must be a complete, standalone insight, not a keyword fragment or a raw quote.
- Prioritize the most impactful/frequent issues and strengths, not minor ones.
- Recommendations must be concrete and actionable, not generic advice like "improve user experience."
- Keep the whole thing concise enough to read in under a minute.
"""

In [120]:
api_key = ""

In [125]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import json

def df_to_review_text(df, n=5):
    if df.empty:
        return "No reviews available."
    lines = []
    for _, row in df.head(n).iterrows():
        content = str(row.get("content", "")).strip().replace("\n", " ")
        thumbs = row.get("thumbsUpCount", 0)
        lines.append(f"- ({thumbs} upvotes) {content}")
    return "\n".join(lines)

USER_PROMPT = """TOPIC_CLUSTERS:
{topic_clusters}

TOP_POSITIVE_REVIEWS:
{top_positive}

TOP_NEUTRAL_REVIEWS:
{top_neutral}

TOP_NEGATIVE_REVIEWS:
{top_negative}
"""

prompt = PromptTemplate(
    template=SYSTEMPROMPT + "\n\n" + USER_PROMPT,
    input_variables=["topic_clusters", "top_positive", "top_neutral", "top_negative"],
)

llm = ChatGroq(
    model_name="openai/gpt-oss-20b",
    groq_api_key=api_key,
    request_timeout=60.0,
    max_retries=3,
)

chain = prompt | llm | StrOutputParser()

result_text = chain.invoke({
    "topic_clusters": json.dumps(summary, ensure_ascii=False),
    "top_positive": df_to_review_text(df_positive),
    "top_neutral": df_to_review_text(df_neutral),
    "top_negative": df_to_review_text(df_negative),
})

print(result_text)

## Overall Summary  
The app remains a popular hub for game saves, achievements, and built‑in casual games, but recent updates have introduced a wave of bugs that frustrate many users. While people praise the tight integration with Google services, they are increasingly upset by crashes, storage‑error messages, and broken playlist functionality.

## What Users Like  
- Seamless transfer of game progress across devices, keeping progress intact after reinstall.  
- Centralized access to a variety of built‑in games that work offline.  
- Integration with Google Fit and calendar features, adding health‑tracking convenience.  
- Friendly, minimal‑data‑usage built‑in games (e.g., Solitaire, Cricket).

## What Users Are Complaining About  
- Frequent crashes and error dialogs after the latest update, especially during playlist editing.  
- Incorrect storage‑requirement prompts (“Insert an SD card…”) even when ample space is available.  
- Auto‑update failures that leave many apps stuck for da